In [1]:
import os
import gc
import copy
import ray
import json
import pickle as pkl
import pandas as pd
from tqdm import tqdm
import random as random
import numpy as np
from ray import tune
import datasets
import numpy as np
from sklearn import metrics
from datasets import load_metric
from torch.utils.data import DataLoader
from utils import get_texts_for_ptype

from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from datasets import load_dataset
from ray.tune.search.hyperopt import HyperOptSearch
from ray.tune.schedulers import ASHAScheduler
import ray.train.huggingface.transformers
from ray.train import ScalingConfig
from ray.train.torch import TorchTrainer

In [15]:
data_dir = "../data"
x_tensor_path = os.path.join(data_dir, "x_tensor.pkl")
with open(x_tensor_path, "rb") as f:
    x_tensor_sample = pkl.load(f)

In [2]:
model_name_types_tuples = [
#     ('BioBERT', 'dmis-lab/biobert-base-cased-v1.2'),
#     ('PubMedBERT', 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'),
    ('DistilBERT', 'distilbert-base-uncased')
]

In [3]:
name = "DistilBERT"

In [27]:
nums = sorted([int(el.split('BERT_')[1].split('_emb')[0]) for el in os.listdir('../data/x_bert')if name in el])

In [28]:
embeddings = []
for num in tqdm(nums):
    direc = f"../data/x_bert/x_{name}_{num}_embeddings.pkl"
    with open(direc, "rb") as f:
        embedding = pkl.load(f)
        embeddings.append(embedding)

100%|███████████████████████████████████████████| 213110/213110 [00:17<00:00, 12325.27it/s]


In [29]:
embeddings = np.concatenate(embeddings)

In [30]:
embeddings.shape

(3409752, 768)

In [31]:
import torch
x_tensor = torch.tensor(embeddings)

In [32]:
del embeddings

In [33]:
with open(f"../data/x_tensor_{name}.pkl", "wb") as f:
    pkl.dump(x_tensor, f)

In [10]:
len(embeddings)

3409752

In [11]:
3409752

3409752

In [15]:
import torch
torch.tensor(embeddings).to(device)

tensor([[ 0.1724, -0.3302, -0.1277,  ...,  0.3337, -0.0842,  0.2375],
        [ 0.0835, -0.1700, -0.0499,  ...,  0.3566, -0.0937,  0.1574],
        [ 0.2820,  0.0311, -0.1996,  ...,  0.4055, -0.1871,  0.1006],
        ...,
        [ 0.1057, -0.2268, -0.2839,  ...,  0.2755, -0.0919,  0.0551],
        [ 0.1258, -0.0579, -0.0953,  ...,  0.2843, -0.3095,  0.1414],
        [ 0.1128,  0.0415, -0.1607,  ...,  0.4949, -0.1837,  0.1741]],
       device='cuda:0')

In [3]:
#     with open("../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl", "rb") as f:
#             prop_typ_id_dict = pkl.load(f)
#     ptype = "is_rct"
#     pos_data_human = pd.read_csv(filename_pos_human, chunksize=100000)
#     neg_data_human = pd.read_csv(filename_neg_human, chunksize=100000)
#     sr_pos_train_ids = prop_typ_id_dict[ptype]["pos"]["train"]
    

In [16]:
with open("../data/pmid2node_id_sample.json", "r") as f:
    pmid2node_id = json.load(f)
pmid2node_id = {int(key): value for key, value in tqdm(pmid2node_id.items())}

100%|█████████████████████████████████████████| 101223/101223 [00:00<00:00, 5141196.82it/s]


In [17]:
pmids_included = list(pmid2node_id)
pmids_included

[384,
 861,
 1469,
 1858,
 2092,
 2404,
 2975,
 5889,
 8070,
 9352,
 9376,
 9954,
 10295,
 11208,
 12805,
 13072,
 14155,
 14362,
 14548,
 15665,
 16701,
 19028,
 19348,
 21786,
 22945,
 23136,
 24330,
 25411,
 26261,
 26697,
 27766,
 28152,
 29755,
 31088,
 31123,
 32907,
 32961,
 33616,
 33710,
 34127,
 34144,
 34151,
 34782,
 35003,
 36459,
 39112,
 39134,
 39895,
 40713,
 40795,
 42207,
 43035,
 43106,
 47348,
 49229,
 50017,
 50295,
 50442,
 51284,
 51587,
 51839,
 51964,
 52001,
 52157,
 52188,
 52767,
 53722,
 57959,
 60260,
 60434,
 61720,
 61994,
 68132,
 69724,
 69727,
 70653,
 71139,
 71267,
 72162,
 72787,
 72830,
 74618,
 76016,
 76069,
 76841,
 78007,
 78872,
 80105,
 81925,
 85168,
 86292,
 86364,
 86524,
 87967,
 88109,
 90013,
 90126,
 91323,
 91397,
 91478,
 91666,
 93874,
 95641,
 95717,
 96461,
 97948,
 98926,
 100008,
 101263,
 102202,
 102447,
 103854,
 104635,
 104771,
 105582,
 106048,
 106428,
 106689,
 107512,
 109121,
 109193,
 109740,
 110520,
 110624,
 1108

In [18]:
len(set(pmids_included))

101223

In [19]:

#     filename_pos_human = "../data/pubmed_articles_pos_human.csv"
#     filename_neg_human = "../data/pubmed_articles_neg_human.csv"


#     pos_data_human = pd.read_csv(filename_pos_human, chunksize=100000)
#     neg_data_human = pd.read_csv(filename_neg_human, chunksize=100000)

#     x_texts = [0 for _ in range(len(pmid2node_id))]
    
#     for pos_chunk in tqdm(pos_data_human):
        
#         for pmid, title, abstract in zip(pos_chunk['pmid'], pos_chunk['title'], pos_chunk['abstract_plaintext']):
#             if pmid in pmid2node_id:
#                 x_texts[pmid2node_id[pmid]] = f"{title} {abstract}"
        

#     for neg_chunk in tqdm(neg_data_human):
#         for pmid, title, abstract in zip(neg_chunk['pmid'], neg_chunk['title'], neg_chunk['abstract_plaintext']):
#             if pmid in pmid2node_id:
#                 x_texts[pmid2node_id[pmid]] = f"{title} {abstract}"

    

19it [00:25,  1.36s/it]/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
51it [01:09,  1.50s/it]/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [01:59,  1.39s/it]
57it [01:28,  1.55s/it]


In [20]:
len([el for el in x_texts if type(el)==int])

0

In [21]:
with open("../data/x_texts_sample.pkl", 'wb') as f:
    pkl.dump(x_texts, f)

In [2]:
# with open("../data/x_texts.pkl", 'rb') as f:
#     x_texts = pkl.load(f)

In [22]:
len(x_texts)

101223

In [10]:
model_ptypes_tuples = [
    ('is_rct', 'dmis-lab/biobert-base-cased-v1.2'),
    ('is_sr', 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'),
    ('is_casecontrol', 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'),
    ('is_cohort', 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'),
    ('is_guideline', 'distilbert-base-uncased')
]

In [23]:
model_name_types_tuples = [
    ('BioBERT', 'dmis-lab/biobert-base-cased-v1.2'),
    ('PubMedBERT', 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'),
    ('DistilBERT', 'distilbert-base-uncased')
]

In [24]:
model_name_types_tuples[0]

('BioBERT', 'dmis-lab/biobert-base-cased-v1.2')

In [25]:
import torch
import numpy as np
def mean_pooling(token_embeddings, mask):
    """
    Applies mean pooling to token embeddings to create sentence embeddings.

    Args:
        token_embeddings: torch.tensor
            token embeddings created using LLM
        mask: torch.tensor
            mask to apply to embeddings

    Returns:
        sentence_embeddings: torch.tensor
            sentence embeddings for information retrieval
    """
    # apply mask to token embeddings
    token_embeddings = token_embeddings.masked_fill(~mask[..., None].bool(), 0.)

    # calculate mean of token embeddings to create sentence embeddings
    sentence_embeddings = token_embeddings.sum(dim=1) / mask.sum(dim=1)[..., None]
    return sentence_embeddings




In [53]:
def create_dataset_tokenized(dataset, tokenizer, device):
    """
    Tokenizes input dataset and allocates data to CPU or GPU.

    Args:
        dataset: list(str)
            question ar answer dataset
        tokenizer: AutoTokenizer
            LLM tokenizer to be applied to dataset
        device: torch.device
            CPU or GPU device

    Returns:
        dataset_tokenized: dict(str) = torch.tensor
            tokenized dataset allocated to device
    """
    # tokenize dataset
    dataset_tokenized = tokenizer(dataset, padding=True, truncation=True, max_length=512)

    # convert dataset to torch dataset and allocate to device
    dataset_tokenized = {
        'input_ids': torch.tensor(dataset_tokenized['input_ids']).to(device),
#         'token_type_ids': torch.tensor(dataset_tokenized['token_type_ids']).to(device),
        'attention_mask': torch.tensor(dataset_tokenized['attention_mask']).to(device),
    }

    return dataset_tokenized

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [28]:
batch_size_outer = 160000
batch_size_inner = 16

num_iter_outer = len(x_texts)//batch_size_outer
num_iter_inner = batch_size_outer//batch_size_inner

In [29]:
name, model_type = model_name_types_tuples[0]
tokenizer = AutoTokenizer.from_pretrained(model_type)
print('tokenizer')
model = AutoModel.from_pretrained(model_type).to(device)

tokenizer


/home/k2141255/anaconda3/envs/study_design_classifier_bert/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [30]:
tokenizer

BertTokenizerFast(name_or_path='dmis-lab/biobert-base-cased-v1.2', vocab_size=28996, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [11]:
dataset_tokenized = tokenizer(list(x_texts[:2]), padding=True, truncation=True, max_length=512)

In [13]:
dataset_tokenized.keys()

dict_keys(['input_ids', 'attention_mask'])

In [17]:
16*num_iter_inner

160000

In [18]:
21*num_iter_inner

210000

In [32]:
for i in tqdm(range(num_iter_outer)):
    print((i)*batch_size_outer, (i+1)*batch_size_outer)

100%|██████████████████████████████████████████████████| 21/21 [00:00<00:00, 211731.69it/s]

0 160000
160000 320000
320000 480000
480000 640000
640000 800000
800000 960000
960000 1120000
1120000 1280000
1280000 1440000
1440000 1600000
1600000 1760000
1760000 1920000
1920000 2080000
2080000 2240000
2240000 2400000
2400000 2560000
2560000 2720000
2720000 2880000
2880000 3040000
3040000 3200000
3200000 3360000


In [46]:
x_texts_tokenized = create_dataset_tokenized(list(x_texts[num_iter_outer*batch_size_outer:]), tokenizer, device)
x_texts_loader = list(DataLoader(list(zip(x_texts_tokenized['input_ids'], x_texts_tokenized['token_type_ids'],
                                                 x_texts_tokenized['attention_mask'])), batch_size=batch_size_inner))


In [54]:
len(x_texts_loader)

3110

In [47]:
len(x_texts[num_iter_outer*batch_size_outer:])

49752

In [ ]:
dataset_tokenized = tokenizer(list(x_texts[i*batch_size_outer:(i+1)*batch_size_outer]), padding=True, truncation=True, max_length=512)

In [ ]:
tokenizer

In [51]:
    # for name, model_type in tqdm(model_name_types_tuples):
    
    name, model_type = model_name_types_tuples[2]
    tokenizer = AutoTokenizer.from_pretrained(model_type)
    print('tokenizer')
    model = AutoModel.from_pretrained(model_type).to(device)
    print('model')
    
    print('loaded')
    x_embeddings = []
    counter = 0
    for i in tqdm(range(num_iter_outer)):
        x_texts_tokenized = create_dataset_tokenized(list(x_texts[i*batch_size_outer:(i+1)*batch_size_outer]), tokenizer, device)
        print((i+1)*batch_size_outer)
        x_texts_loader = DataLoader(list(zip(x_texts_tokenized['input_ids'], 
#                                              x_texts_tokenized['token_type_ids'],
                                             x_texts_tokenized['attention_mask'])), batch_size=batch_size_inner)
#         for input_ids, token_type_ids, attention_mask in tqdm(x_texts_loader):
        for input_ids, attention_mask in tqdm(x_texts_loader):
            batch_x_tok = {
                'input_ids': input_ids,
#                 'token_type_ids': token_type_ids,
                'attention_mask': attention_mask
            }
            x_outputs = model(**batch_x_tok)
            
            x_embedding = mean_pooling(x_outputs[0], mask=attention_mask).cpu().detach().numpy()
            with open(f'../data/x_bert_sample/x_{name}_{counter}_embeddings.pkl', 'wb') as f:
                pkl.dump(x_embedding, f)
            
            x_embeddings.append(x_embedding)
            counter +=1
            

 

tokenizer
model
loaded


0it [00:00, ?it/s]


In [54]:
x_texts_tokenized = create_dataset_tokenized(list(x_texts[num_iter_outer*batch_size_outer:]), tokenizer, device)
x_texts_loader = DataLoader(list(zip(x_texts_tokenized['input_ids'], 
#                                      x_texts_tokenized['token_type_ids'],
                                                 x_texts_tokenized['attention_mask'])), batch_size=batch_size_inner)
# for input_ids, token_type_ids, attention_mask in tqdm(x_texts_loader):
for input_ids, attention_mask in tqdm(x_texts_loader):
    batch_x_tok = {
        'input_ids': input_ids,
#         'token_type_ids': token_type_ids,
        'attention_mask': attention_mask
    }
    x_outputs = model(**batch_x_tok)
    x_embedding = mean_pooling(x_outputs[0], mask=attention_mask).cpu().detach().numpy()
    with open(f'../data/x_bert_sample/x_{name}_{counter}_embeddings.pkl', 'wb') as f:
        pkl.dump(x_embedding, f)
    x_embeddings.append(x_embedding)
#     counter +=1

100%|██████████████████████████████████████████████████| 6327/6327 [05:53<00:00, 17.89it/s]


In [55]:
x_embeddings = np.concatenate(x_embeddings, axis=0)
print('concat')

concat


In [56]:
x_embeddings.shape

(101223, 768)

In [57]:
x_tensor_sample = torch.tensor(x_embeddings)

In [58]:
x_tensor_sample.shape

torch.Size([101223, 768])

In [59]:
with open(f"../data/x_tensor_{name}_sample", "wb") as f:
    pkl.dump(x_tensor_sample, f)

In [60]:
name

'DistilBERT'

In [ ]:
# with open(f'../data/x_{name}_embeddings.pkl', 'wb') as f:
#     pkl.dump(x_embeddings, f)
#     print('save')

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
np.concatenate(x_embeddings, axis=0)

In [ ]:
import pickle as pkl
name = "BioBERT"
with open(f'../data/x_{name}_embeddings.pkl', 'rb') as f:
        x_embeddings = pkl.load(f)
        print('save')

In [ ]:
model_name_types_tuples = [
    ('BioBERT', 'dmis-lab/biobert-base-cased-v1.2'),
    ('PubMedBERT' 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'),
    ('DistilBERT', 'distilbert-base-uncased')
]